In [8]:
import cv2
import numpy as np
from tensorflow import keras
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from cvzone.HandTrackingModule import HandDetector
from collections import Counter

# ==========================================
# HAND SIGN DETECTOR
# ==========================================
class HandSignDetector:

    def __init__(
        self,
        model_path="sign_alphabet_mob_model.keras",
        class_indices_path="class_indices_mob.npy"
    ):

        print("Loading MobileNetV2 model...")

        self.model = keras.models.load_model(
            model_path,
            compile=False
        )

        print("Model loaded successfully!")

        class_indices = np.load(
            class_indices_path,
            allow_pickle=True
        ).item()

        self.idx_to_class = {
            v: k for k, v in class_indices.items()
        }

        print("\nClass Mapping:")
        print(self.idx_to_class)

        self.detector = HandDetector(
            maxHands=2,
            detectionCon=0.5
        )

        self.img_size = 224
        self.prediction_buffer = []

    # ==========================================
    # PREPROCESS IMAGE
    # ==========================================
    def preprocess_image(self, img_crop):

        target_size = self.img_size
    
        # Create black canvas
        canvas = np.zeros(
            (target_size, target_size, 3),
            dtype=np.uint8
        )
    
        h, w = img_crop.shape[:2]
    
        if h == 0 or w == 0:
            return None, None
    
        # Preserve aspect ratio
        scale = min(
            target_size / w,
            target_size / h
        )
    
        new_w = int(w * scale)
        new_h = int(h * scale)
    
        resized = cv2.resize(
            img_crop,
            (new_w, new_h)
        )
    
        x_offset = (target_size - new_w) // 2
        y_offset = (target_size - new_h) // 2
    
        canvas[
            y_offset:y_offset + new_h,
            x_offset:x_offset + new_w
        ] = resized
    
        # MobileNetV2 preprocessing
        img_preprocessed = preprocess_input(
            canvas.astype(np.float32)
        )
    
        img_input = np.expand_dims(
            img_preprocessed,
            axis=0
        )
    
        return img_input, canvas    
    # ==========================================
    # DETECT + PREDICT
    # ==========================================
    def predict(self, img):

        try:

            result = self.detector.findHands(
                img,
                draw=False
            )

            if isinstance(result, tuple):
                hands, img = result
            else:
                hands = result

            if hands is None or len(hands) == 0:
                return None, 0, None, None

            # ==========================================
            # LANDMARK BASED CROP
            # ==========================================
            all_x = []
            all_y = []

            for hand in hands:
                #x, y, w, h = hand['bbox']

                #all_x.extend([x, x+w])
                #all_y.extend([y, y+h])

                lm_list = hand["lmList"]
                

                for lm in lm_list:
            

                    all_x.append(int(lm[0]))
                    all_y.append(int(lm[1]))

            pad = 70

            frame_h, frame_w = img.shape[:2]

            x1 = max(0, min(all_x) - pad)
            y1 = max(0, min(all_y) - pad)

            x2 = min(frame_w, max(all_x) + pad)
            y2 = min(frame_h, max(all_y) + pad)

            img_crop = img[
                y1:y2,
                x1:x2
            ]

            if img_crop.size == 0:
                return None, 0, None, None

            img_input, processed_img = self.preprocess_image(
                img_crop
            )

            print("Crop shape:", img_crop.shape)

            if img_input is None:

                return None, 0, None, None

            prediction = self.model.predict(
                img_input,
                verbose=0
            )

            index = np.argmax(prediction[0])

            confidence = float(
                prediction[0][index]
            )

            # Ignore weak predictions
            #if confidence < 0.50:
                #return (
                    #None,
                    #confidence,
                    #(x1, y1, x2, y2),
                    #processed_img
                #)

            label = self.idx_to_class[index]

            # ==========================================
            # SMOOTH PREDICTIONS
            # ==========================================
            self.prediction_buffer.append(label)

            if len(self.prediction_buffer) > 5:
                self.prediction_buffer.pop(0)

            label = Counter(
                self.prediction_buffer
            ).most_common(1)[0][0]

            # Debug Top-3
            top3 = np.argsort(
                prediction[0]
            )[-3:][::-1]

            print("\nTop 3 Predictions:")

            for idx in top3:

                print(
                    f"{self.idx_to_class[idx]} : "
                    f"{prediction[0][idx]:.4f}"
                )

            return (
                label,
                confidence,
                (x1, y1, x2, y2),
                processed_img
            )

        except Exception as e:

            print("Prediction Error:", e)

            return None, 0, None, None


# ==========================================
# MAIN
# ==========================================
def main():

    cap = cv2.VideoCapture(0)

    cap.set(
        cv2.CAP_PROP_FRAME_WIDTH,
        1280
    )

    cap.set(
        cv2.CAP_PROP_FRAME_HEIGHT,
        720
    )

    detector = HandSignDetector()

    while True:

        success, img = cap.read()

        if not success:
            continue

        img = cv2.flip(img, 1)

        (
            label,
            confidence,
            bbox,
            processed_img
        ) = detector.predict(img)

        if bbox is not None:

            x1, y1, x2, y2 = bbox

            cv2.rectangle(
                img,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

        if label is not None:

            cv2.putText(
                img,
                f"{label} ({confidence:.2f})",
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
            )

        if processed_img is not None:

            cv2.imshow(
                "Model Input",
                processed_img
            )

        cv2.imshow(
            "Two-Hand Sign Recognition",
            img
        )

        key = cv2.waitKey(1)

        if key == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# ==========================================
# RUN
# ==========================================
if __name__ == "__main__":
    main()

Loading MobileNetV2 model...
Model loaded successfully!

Class Mapping:
{0: 'a', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}
Crop shape: (254, 446, 3)

Top 3 Predictions:
b : 0.8712
w : 0.0796
j : 0.0471
Crop shape: (246, 364, 3)

Top 3 Predictions:
m : 0.6686
j : 0.1965
f : 0.1054
Crop shape: (240, 404, 3)

Top 3 Predictions:
j : 0.8277
a : 0.1504
g : 0.0114
Crop shape: (235, 483, 3)

Top 3 Predictions:
w : 0.9111
m : 0.0842
j : 0.0038
Crop shape: (235, 512, 3)

Top 3 Predictions:
w : 0.9985
h : 0.0005
a : 0.0005
Crop shape: (241, 521, 3)

Top 3 Predictions:
m : 0.6784
w : 0.2273
j : 0.0521
Crop shape: (233, 537, 3)

Top 3 Predictions:
w : 0.9645
s : 0.0143
g : 0.0116
Crop shape: (236, 532, 3)

Top 3 Predictions:
w : 0.9998
m : 0.0001
j : 0.0001
Crop shape: (247, 502, 3)

Top 3 Predictions:
m : 0.9938
w : 0.0062
j :